# M6b · Build the three marts views

**Goal:** (re)create `marts.v_device_risk_profile`, `marts.v_ml_features_driver_behavior`, `marts.v_fleet_risk_dashboard`.

Views are CREATE-OR-REPLACE: always idempotent.

**Exit criterion:** each view returns rows.


In [ ]:
# === Setup ===
from __future__ import annotations
import sys, pathlib
# Make the src/ package importable even if pip install -e was skipped.
PROJECT_ROOT = pathlib.Path().resolve().parents[1] if pathlib.Path().resolve().name != 'accent-fleet-analytics' else pathlib.Path().resolve()
for candidate in (PROJECT_ROOT, PROJECT_ROOT.parent):
    src = candidate / 'src'
    if src.exists() and str(src) not in sys.path:
        sys.path.insert(0, str(src))
        break

from accent_fleet.config import settings
from accent_fleet.db import get_engine, transaction
from sqlalchemy import text

s = settings()
print(f"DB = {s.pg_user}@{s.pg_host}:{s.pg_port}/{s.pg_database}")
print(f"Schemas: staging={s.pg_schema_staging}, warehouse={s.pg_schema_warehouse}, marts={s.pg_schema_marts}")


## 2. Inputs

In [ ]:
VIEW_SQL_FILES = [
    '21_v_device_risk_profile.sql',
    '22_v_ml_features.sql',
    '23_v_fleet_risk_dashboard.sql',
]


## 3. Execute

In [ ]:
from accent_fleet.db import run_sql_file
with transaction() as conn:
    for f in VIEW_SQL_FILES:
        run_sql_file(conn, f)
        print('applied', f)


## 4. Inspect

In [ ]:
import pandas as pd
with get_engine().connect() as conn:
    for v in ['v_device_risk_profile','v_ml_features_driver_behavior','v_fleet_risk_dashboard']:
        n = conn.execute(text(f'SELECT COUNT(*) FROM marts.{v}')).scalar_one()
        print(f'marts.{v}: {n:,} rows')
    display(pd.read_sql(text('SELECT * FROM marts.v_device_risk_profile ORDER BY risk_score DESC LIMIT 10'), conn))
